In [10]:
!pip install dagshub mlflow

In [11]:
import dagshub
import mlflow
import skops.io as sio
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import Ridge
from sklearn.feature_selection import RFE

dagshub.init(repo_owner='ZukaCS', repo_name='house-prices-ML-assignment1', mlflow=True)


Initialized MLflow to track repo "ZukaCS/house-prices-ML-assignment1"

Repository ZukaCS/house-prices-ML-assignment1 initialized!

In [12]:
from sklearn.base import BaseEstimator, TransformerMixin

class HouseHandleNaN(BaseEstimator, TransformerMixin):
    cat_na_cols = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                   'BsmtCond', 'BsmtFinType1', 'BsmtExposure', 'BsmtQual', 'BsmtFinType2',
                   'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'MasVnrType']
    num_na_cols = ['GarageYrBlt']
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        for col in self.cat_na_cols:
            if col in X.columns:
                X[col] = X[col].fillna("None")
        for col in self.num_na_cols:
            if col in X.columns:
                X[col] = X[col].fillna(0)

        
        return X

In [13]:
class OrdinalEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        
        quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
        quality_cols = ['GarageQual', 'GarageCond', 'PoolQC', 'ExterQual',
                        'ExterCond', 'BsmtCond', 'HeatingQC', 'KitchenQual',
                        'BsmtQual', 'FireplaceQu']
        
        for col in quality_cols:
            if col in X.columns:
                X[col] = X[col].map(quality_map)
        
        if 'GarageFinish' in X.columns:
            X['GarageFinish'] = X['GarageFinish'].map({'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3})
        
        if 'PavedDrive' in X.columns:
            X['PavedDrive'] = X['PavedDrive'].map({'N': 0, 'P': 1, 'Y': 2})
        
        if 'Functional' in X.columns:
            X['Functional'] = X['Functional'].map(
                {'Sal': 1, 'Sev': 2, 'Maj2': 3, 'Maj1': 4, 'Mod': 5, 'Min2': 6, 'Min1': 7, 'Typ': 8})
        
        if 'LandSlope' in X.columns:
            X['LandSlope'] = X['LandSlope'].map({'Sev': 1, 'Mod': 2, 'Gtl': 3})
        return X

In [14]:
class HouseFeatureAdderAndModifier(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        
        # make total areas
        X["TotalSF"]  = X["TotalBsmtSF"] + X["1stFlrSF"] + X["2ndFlrSF"]
        X["TotalPorchSF"] = X["OpenPorchSF"] + X["EnclosedPorch"] + X["3SsnPorch"] + X["ScreenPorch"]

        X["TotalBath"] = X["FullBath"] + X["BsmtFullBath"] + 0.4*X["HalfBath"] + 0.4*X["BsmtHalfBath"] #maybe people like Full baths more and less halfs
        X["HouseAge"]   = X["YrSold"] - X["YearBuilt"]
        X["RemodelAge"] = X["YrSold"] - X["YearRemodAdd"]
        
        X["BsmtFinTotal"] = X["BsmtFinSF1"] + X["BsmtFinSF2"]
     
        X["HasGarage"]    = (X["GarageArea"] > 0).astype(int)
        X["HasBasement"]  = (X["TotalBsmtSF"] > 0).astype(int)
        X["HasPool"]      = (X["PoolArea"] > 0).astype(int)
        X["Has2ndFloor"]  = (X["2ndFlrSF"] > 0).astype(int)
        X["WasRemodeled"] = (X["YearRemodAdd"] != X["YearBuilt"]).astype(int)
        
        # drop things we used to make other columns
        X = X.drop(columns=[
            "1stFlrSF", "2ndFlrSF",          
            "OpenPorchSF", "EnclosedPorch",   
            "3SsnPorch", "ScreenPorch",       
            "FullBath", "HalfBath",           
            "BsmtFullBath", "BsmtHalfBath",   
            "BsmtFinSF1", "BsmtFinSF2",     
            "YearBuilt", "YearRemodAdd",      
        ])
        
        return X

In [15]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.85):
        self.threshold = threshold
        self.selected_features_ = None

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        corr_matrix = X.corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        self.features_to_drop_ = set()
        for col in upper.columns:
            if any(upper[col] > self.threshold):
                partners = upper.index[upper[col] > self.threshold].tolist()
                for partner in partners:
                    if corr_matrix[col].mean() >= corr_matrix[partner].mean():
                        self.features_to_drop_.add(col)
                    else:
                        self.features_to_drop_.add(partner)
        self.selected_features_ = [
            col for col in X.columns if col not in self.features_to_drop_
        ]
        print(f"Dropped {len(self.features_to_drop_)} correlated features")
        print(f"Remaining: {len(self.selected_features_)} features")
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return X[self.selected_features_].values

In [16]:
from sklearn.linear_model import Ridge
from sklearn.feature_selection import RFE

class RFESelector(BaseEstimator, TransformerMixin):
    def __init__(self, n_features_to_select=50, estimator=None):
        self.n_features_to_select = n_features_to_select
        self.estimator = estimator

    def fit(self, X, y):
        X = pd.DataFrame(X).reset_index(drop=True)
        self.feature_names_ = list(X.columns)
        est = self.estimator if self.estimator is not None else Ridge(alpha=10)
        self.rfe_ = RFE(estimator=est, n_features_to_select=self.n_features_to_select)
        self.rfe_.fit(X, y)
        self.selected_features_ = [
            f for f, s in zip(self.feature_names_, self.rfe_.support_) if s
        ]
        print(f"RFE selected: {len(self.selected_features_)} features")
        return self

    def transform(self, X):
        X = pd.DataFrame(X).reset_index(drop=True)
        X.columns = self.feature_names_
        return X[self.selected_features_].values

In [17]:

preprocessing_run_id = "41e1495e8bad49a4a29e36d99b2f537f"
best_run_id          = "5f1bf0c0f3744289afd73b9ff316d3d6"

trusted_types = [
    "__main__.HouseHandleNaN",
    "__main__.HouseFeatureAdderAndModifier",
    "__main__.OrdinalEncoder",
    "__main__.CorrelationFilter",
    "__main__.RFESelector",
    "sklearn.pipeline.Pipeline",
    "sklearn.compose._column_transformer.ColumnTransformer",
    "sklearn.compose._column_transformer._RemainderColsList",
    "sklearn.impute._base.SimpleImputer",
    "sklearn.preprocessing._encoders.OneHotEncoder",
    "sklearn.preprocessing._data.StandardScaler",
    "sklearn.linear_model._ridge.Ridge",
    "sklearn.linear_model._coordinate_descent.Lasso",
    "sklearn.feature_selection._rfe.RFE",
    "numpy.dtype",
]


pipeline_path = mlflow.artifacts.download_artifacts(
    run_id=preprocessing_run_id,
    artifact_path="preprocessing_pipeline_linear.skops"
)
pipeline = sio.load(pipeline_path, trusted=trusted_types)
print("Pipeline loaded!")


model_path = mlflow.artifacts.download_artifacts(
    run_id=best_run_id,
    artifact_path="model.skops"
)
model = sio.load(model_path, trusted=trusted_types)
print("Model loaded!")
print(type(model))


PATH = "/kaggle/input/competitions/house-prices-advanced-regression-techniques/"
test_df = pd.read_csv(PATH + "test.csv")
passenger_ids = test_df["Id"]

X_kaggle = pipeline.transform(test_df)
X_kaggle = pd.DataFrame(X_kaggle).fillna(0).values

kaggle_preds = np.expm1(model.predict(X_kaggle))

submission = pd.DataFrame({
    "Id":        passenger_ids,
    "SalePrice": kaggle_preds
})
submission.to_csv("submission.csv", index=False)
print(submission.shape)
print(submission.head())

Pipeline loaded!


Model loaded!
<class 'sklearn.linear_model._coordinate_descent.Lasso'>
(1459, 2)
     Id      SalePrice
0  1461  124665.139598
1  1462  153929.126080
2  1463  176851.844718
3  1464  194932.766053
4  1465  198453.828951
